In [1]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
torch.set_float32_matmul_precision('high')  # utilize Tensor Cores on RTX 3090

import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import RMSE, SMAPE

In [2]:
Y_COL    = 'Sum of кВт'
ID_COL   = 'EIC-код_cat'
DS_COL   = 'datetime'

WEATHER_COLS = [
    'temperature_2m', 'apparent_temperature', 'dew_point_2m',
    'relative_humidity_2m', 'precipitation', 'rain', 'snowfall',
    'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high',
    'surface_pressure', 'wind_speed_10m', 'wind_direction_10m',
    'wind_gusts_10m', 'shortwave_radiation', 'diffuse_radiation',
    'direct_normal_irradiance',
]
TIME_COLS  = ['month', 'hour', 'dow', 'season']   # numeric versions, created in prepare_nf
FUTR_EXOG  = WEATHER_COLS + TIME_COLS
STAT_EXOG  = ['lat', 'lon']                        # per-location static features

HORIZON    = 48    # ~1 month of hourly steps  (used for val/test evaluation)
INPUT_SIZE = 4 * HORIZON  # 3 months of context (was 2)

# Trim each series to the last KEEP_HOURS steps before fitting.
# NeuralForecast dataset init is CPU-bound O(N) — trimming from ~12k→4380h
# per series cuts init time ~3x with negligible accuracy loss.
KEEP_HOURS = 6 * 30 * 24  # ~6 months per series (~4380 hours)

In [3]:
def smallest_int_dtype(min_val, max_val, signed=True):
    if signed:
        for dtype in ['int8', 'int16', 'int32', 'int64']:
            info = np.iinfo(dtype)
            if info.min <= min_val <= max_val <= info.max:
                return dtype
    else:
        for dtype in ['uint8', 'uint16', 'uint32', 'uint64']:
            info = np.iinfo(dtype)
            if 0 <= min_val <= max_val <= info.max:
                return dtype
    return 'int64'


def optimize_df_for_memory(df):
    meta = {}
    for col in df.columns:
        s = df[col]
        unique_non_null = set(s.dropna().unique())
        if unique_non_null.issubset({0, 1, True, False}) and col == 'Група':
            df[col] = s.astype('bool')
            meta[col] = {'stored_as': 'bool', 'scale': 1}
            continue
        if pd.api.types.is_integer_dtype(s):
            mn, mx = int(s.min()), int(s.max())
            dtype = smallest_int_dtype(mn, mx, signed=(mn < 0))
            df[col] = s.astype(dtype)
            meta[col] = {'stored_as': dtype, 'scale': 1}
            continue
        if pd.api.types.is_float_dtype(s):
            non_null = s.dropna()
            if len(non_null) == 0:
                df[col] = s.astype('float32')
                meta[col] = {'stored_as': 'float32', 'scale': 1}
                continue
            decimals = non_null.astype(str).apply(
                lambda x: len(x.split('.')[1].rstrip('0')) if '.' in x else 0
            ).max()
            if decimals <= 3:
                scale  = 10 ** decimals
                scaled = np.round(s * scale)
                mn, mx = int(np.nanmin(scaled)), int(np.nanmax(scaled))
                int_dtype = smallest_int_dtype(mn, mx, signed=(mn < 0))
                if np.dtype(int_dtype).itemsize < np.dtype('float32').itemsize:
                    df[col] = scaled.astype(int_dtype)
                    meta[col] = {'stored_as': int_dtype, 'scale': scale}
                    continue
            df[col] = s.astype('float32')
            meta[col] = {'stored_as': 'float32', 'scale': 1}
    return df, meta


def add_cat_helpers(df):
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['Month_cat']       = df['datetime'].dt.month.astype(str)
    df['Day_cat']         = df['datetime'].dt.day.astype(str)
    df['Hour_cat']        = df['datetime'].dt.hour.astype(str)
    df['day_of_week_cat'] = df['datetime'].dt.dayofweek.astype(str)
    season_map = {12: 'winter', 1: 'winter', 2: 'winter',
                   3: 'spring', 4: 'spring', 5: 'spring',
                   6: 'summer', 7: 'summer', 8: 'summer',
                   9: 'autumn', 10: 'autumn', 11: 'autumn'}
    df['season_cat'] = df['datetime'].dt.month.map(season_map)
    return df


def load_and_prepare(path):
    df = pd.read_parquet(path).reset_index(drop=True)
    df.columns = df.columns.str.replace('.', '_', regex=False)
    df, _ = optimize_df_for_memory(df)
    df = add_cat_helpers(df)
    for col in df.columns:
        if col.endswith('_cat'):
            df[col] = df[col].astype(str)
    try:
        df[Y_COL] = df[Y_COL].astype('float32')
    except Exception:
        print(f'No Y_col: {Y_COL}')
    df = df.sort_values([ID_COL, DS_COL]).reset_index(drop=True)
    try:
        df.drop(columns=['Ціна розподілу ЕЕ', 'Ціна ЕЕ', 'Money_spent'], inplace=True)
    except Exception:
        pass
    return df


def smape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    return float(np.mean(np.abs(y_true - y_pred) / np.maximum(denom, 1e-8)))

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mape(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)

In [4]:
SEASON_MAP = {'winter': 0, 'spring': 1, 'summer': 2, 'autumn': 3}


def prepare_nf(df, has_y=True):
    """Convert a raw DataFrame to neuralforecast format."""
    df = df.copy()

    # Normalise datetime: strip timezone if present
    ds = pd.to_datetime(df[DS_COL])
    if ds.dt.tz is not None:
        ds = ds.dt.tz_convert(None)
    df['ds'] = ds

    df = df.rename(columns={ID_COL: 'unique_id'})
    if has_y:
        df = df.rename(columns={Y_COL: 'y'})

    # Numeric time features (neuralforecast requires numeric exog)
    df['month']  = df['Month_cat'].astype(int)
    df['hour']   = df['Hour_cat'].astype(int)
    df['dow']    = df['day_of_week_cat'].astype(int)
    df['season'] = df['season_cat'].map(SEASON_MAP).astype(int)

    # Ensure all weather cols are float32
    for col in WEATHER_COLS:
        df[col] = df[col].astype('float32')

    keep = ['unique_id', 'ds'] + (['y'] if has_y else []) + FUTR_EXOG
    return df[keep].sort_values(['unique_id', 'ds']).reset_index(drop=True)

In [5]:
train = load_and_prepare('data/silver_money_calc/train.parquet')
val   = load_and_prepare('data/silver_money_calc/val.parquet')
test  = load_and_prepare('data/silver_money_calc/test.parquet')

train_nf = prepare_nf(train)
val_nf   = prepare_nf(val)
test_nf  = prepare_nf(test)

print('train:', train_nf.shape, ' val:', val_nf.shape, ' test:', test_nf.shape)

train: (4864324, 25)  val: (304499, 25)  test: (304152, 25)


In [6]:
# Trim to the most recent KEEP_HOURS per series before any fitting.
# This cuts dataset init time from ~1h to a few minutes.
def trim_nf(df, keep_hours=KEEP_HOURS):
    return (
        df.groupby('unique_id', group_keys=False)
        .apply(lambda g: g.iloc[-keep_hours:])
        .reset_index(drop=True)
    )

train_nf_trimmed = trim_nf(train_nf)
print(f'train_nf: {train_nf.shape}  →  trimmed: {train_nf_trimmed.shape}')

train_nf: (4864324, 25)  →  trimmed: (1748965, 25)


C:\Users\Lev\AppData\Local\Temp\ipykernel_21832\1081633619.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby('unique_id', group_keys=False)


In [7]:
# One row per unique_id with static numeric features
static_df = (
    train[[ID_COL, 'Широта', 'Довгота']]
    .drop_duplicates(ID_COL)
    .rename(columns={ID_COL: 'unique_id', 'Широта': 'lat', 'Довгота': 'lon'})
    .reset_index(drop=True)
)
static_df[['lat', 'lon']] = static_df[['lat', 'lon']].astype('float32')
print(static_df.shape)
static_df.head()

(410, 3)


,unique_id,lat,lon
0,62Z0008583037334,48.442429,22.192190
1,62Z0011230718431,51.542107,31.262997
2,62Z0096677872985,48.569660,22.346380
3,62Z0101426517156,48.567394,30.232208
4,62Z013852333354Y,48.566059,30.230684


In [ ]:
def make_nhits(h):
    """Build an N-HiTS model for a given forecast horizon h."""
    return NHITS(
        h=h,
        input_size=INPUT_SIZE,
        futr_exog_list=FUTR_EXOG,
        stat_exog_list=STAT_EXOG,
        # Multi-scale pooling: weekly (168h) -> daily (24h) -> hourly (1h)
        n_blocks=[3, 3, 3],
        mlp_units=[[512, 512], [512, 512], [512, 512]],
        n_pool_kernel_size=[168, 24, 1],
        n_freq_downsample=[168, 24, 1],
        batch_size=32,
        windows_batch_size=256,
        step_size=24,
        learning_rate=5e-4,
        max_steps=2000,
        val_check_steps=100,
        early_stop_patience_steps=10,
        scaler_type='robust',
        loss=SMAPE(),
    )

## Evaluation on Val and Test

Single fit on `train`; context is extended to `train+val` when predicting test.
No second fit needed — the model weights are reused with a longer context window.

In [9]:
# Fit once on trimmed train data; val_size sets aside the last HORIZON steps for early stopping
nf_eval = NeuralForecast(models=[make_nhits(HORIZON)], freq='h')
nf_eval.fit(df=train_nf_trimmed, static_df=static_df, val_size=HORIZON)

Seed set to 1
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
C:\Users\Lev\Miniconda3\envs\Diploma\lib\site-packages\torch\nn\modules\module.py:1329: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ SMAPE         │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ blocks       │ ModuleList    │ 16.9 M │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 16.9 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 16.9 M                                                                                               
Total estimated model params size (MB): 67                                                                         
Modules in train mode: 94                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=2000` reached.


In [10]:
def build_futr_df(context_df, exog_source_df, h, exog_cols):
    """
    Build futr_df with exactly h steps per unique_id starting right after
    context_df ends. Merges exog features from exog_source_df; fills any
    gaps with forward/back fill so NeuralForecast never sees missing rows.
    """
    last_ts = context_df.groupby("unique_id")["ds"].max()
    frames = []
    for uid, last in last_ts.items():
        future_ds = pd.date_range(last + pd.Timedelta(hours=1), periods=h, freq="h")
        frames.append(pd.DataFrame({"unique_id": uid, "ds": future_ds}))
    futr_base = pd.concat(frames, ignore_index=True)
    futr = futr_base.merge(
        exog_source_df[["unique_id", "ds"] + exog_cols],
        on=["unique_id", "ds"], how="left",
    )
    futr[exog_cols] = futr.groupby("unique_id")[exog_cols].ffill().bfill()
    return futr

In [11]:
# Filter context to only series present in val (train has more series than val)
val_ids  = val_nf['unique_id'].unique()
ctx      = train_nf_trimmed[train_nf_trimmed['unique_id'].isin(val_ids)].reset_index(drop=True)
val_exog = val_nf[['unique_id', 'ds'] + FUTR_EXOG]
val_end  = val_nf.groupby('unique_id')['ds'].max()

n_val_est = int(np.ceil(val_nf.groupby('unique_id').size().max() / HORIZON))
print(f'Recursive val prediction: ~{n_val_est} windows of {HORIZON}h')

val_preds = []
step = 0

while step < n_val_est + 2:   # +2 as a safety cap
    ctx_end    = ctx.groupby('unique_id')['ds'].max()
    # Series whose context hasn't yet reached the last val timestamp
    active_ids = ctx_end.index[ctx_end < val_end.reindex(ctx_end.index)].tolist()
    if not active_ids:
        break

    step_ctx  = ctx[ctx['unique_id'].isin(active_ids)].reset_index(drop=True)
    step_stat = static_df[static_df['unique_id'].isin(active_ids)].reset_index(drop=True)
    # build_futr_df generates the exact timestamps NeuralForecast expects
    # (last_ctx_ts + 1h … last_ctx_ts + HORIZON*1h) per series
    step_futr = build_futr_df(step_ctx, val_exog, HORIZON, FUTR_EXOG)

    step_pred = nf_eval.predict(df=step_ctx, static_df=step_stat, futr_df=step_futr)
    val_preds.append(step_pred)

    new_rows = step_futr.merge(
        step_pred.rename(columns={'NHITS': 'y'})[['unique_id', 'ds', 'y']],
        on=['unique_id', 'ds'],
    )
    ctx = (
        pd.concat([ctx, new_rows[ctx.columns]])
        .sort_values(['unique_id', 'ds'])
        .reset_index(drop=True)
    )
    step += 1
    print(f'  Step {step}/{n_val_est} done — active series: {len(active_ids)}, context rows: {len(ctx):,}')

val_pred_df = (
    pd.concat(val_preds)
    .sort_values(['unique_id', 'ds'])
    .reset_index(drop=True)
)
# Inner merge restricts to timestamps that actually exist in val_nf
val_merged = val_nf[['unique_id', 'ds', 'y']].merge(val_pred_df, on=['unique_id', 'ds'])

print(f'\nValidation metrics  ({len(val_merged):,} rows)')
print('SMAPE:', smape(val_merged['y'], val_merged['NHITS']))
print('RMSE :', rmse( val_merged['y'], val_merged['NHITS']))
print('MAPE :', mape( val_merged['y'], val_merged['NHITS']), '%')

Recursive val prediction: ~16 windows of 48h


Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

  Step 1/16 done — active series: 397, context rows: 1,723,358


Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...

Predicting: |          | 0/? [00:00<?, ?it/s]

  Step 15/16 done — active series: 397, context rows: 1,990,142


Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

  Step 16/16 done — active series: 397, context rows: 2,009,198

Validation metrics  (295,368 rows)
SMAPE: 0.18048686940496553
RMSE : 10.043950202432573
MAPE : 18.933172707716842 %


In [13]:
train_val_nf = (
    pd.concat([train_nf, val_nf])
    .sort_values(['unique_id', 'ds'])
    .reset_index(drop=True)
)
# Trim for prediction context (model only uses INPUT_SIZE hours of lookback anyway)
train_val_nf_trimmed = trim_nf(train_val_nf)
print(f'train+val: {train_val_nf.shape}  →  trimmed: {train_val_nf_trimmed.shape}')

train+val: (5168823, 25)  →  trimmed: (1754153, 25)


C:\Users\Lev\AppData\Local\Temp\ipykernel_21832\1081633619.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby('unique_id', group_keys=False)


In [14]:
# Filter context to only series present in test
test_ids  = test_nf['unique_id'].unique()
ctx       = train_val_nf_trimmed[train_val_nf_trimmed['unique_id'].isin(test_ids)].reset_index(drop=True)
test_exog = test_nf[['unique_id', 'ds'] + FUTR_EXOG]
test_end  = test_nf.groupby('unique_id')['ds'].max()

n_test_est = int(np.ceil(test_nf.groupby('unique_id').size().max() / HORIZON))
print(f'Recursive test prediction: ~{n_test_est} windows of {HORIZON}h')

test_preds = []
step = 0

while step < n_test_est + 2:   # +2 as a safety cap
    ctx_end    = ctx.groupby('unique_id')['ds'].max()
    active_ids = ctx_end.index[ctx_end < test_end.reindex(ctx_end.index)].tolist()
    if not active_ids:
        break

    step_ctx  = ctx[ctx['unique_id'].isin(active_ids)].reset_index(drop=True)
    step_stat = static_df[static_df['unique_id'].isin(active_ids)].reset_index(drop=True)
    step_futr = build_futr_df(step_ctx, test_exog, HORIZON, FUTR_EXOG)

    step_pred = nf_eval.predict(df=step_ctx, static_df=step_stat, futr_df=step_futr)
    test_preds.append(step_pred)

    new_rows = step_futr.merge(
        step_pred.rename(columns={'NHITS': 'y'})[['unique_id', 'ds', 'y']],
        on=['unique_id', 'ds'],
    )
    ctx = (
        pd.concat([ctx, new_rows[ctx.columns]])
        .sort_values(['unique_id', 'ds'])
        .reset_index(drop=True)
    )
    step += 1
    print(f'  Step {step}/{n_test_est} done — active series: {len(active_ids)}, context rows: {len(ctx):,}')

test_pred_df = (
    pd.concat(test_preds)
    .sort_values(['unique_id', 'ds'])
    .reset_index(drop=True)
)
# Inner merge restricts to timestamps that actually exist in test_nf
test_merged = test_nf[['unique_id', 'ds', 'y']].merge(test_pred_df, on=['unique_id', 'ds'])

print(f'\nTest metrics  ({len(test_merged):,} rows)')
print('SMAPE:', smape(test_merged['y'], test_merged['NHITS']))
print('RMSE :', rmse( test_merged['y'], test_merged['NHITS']))
print('MAPE :', mape( test_merged['y'], test_merged['NHITS']), '%')

Recursive test prediction: ~16 windows of 48h


Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

  Step 1/16 done — active series: 397, context rows: 1,728,546


Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...

Predicting: |          | 0/? [00:00<?, ?it/s]

  Step 15/16 done — active series: 395, context rows: 1,993,986


Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

  Step 16/16 done — active series: 395, context rows: 2,012,946

Test metrics  (294,277 rows)
SMAPE: 0.17953711653216486
RMSE : 10.556691395287258
MAPE : 29.084511923893203 %


## 6-Month Prediction on `predict_X` (Recursive)

Retrain on **all** labelled data (train + val + test) with `h=HORIZON` (1 month), then
roll forward one month at a time for 6 steps.  
Each step feeds the previous predictions back as context for the next step.

In [15]:
predict_X_raw = load_and_prepare('data/no_y_col/with_weather_v2.parquet')
predict_nf    = prepare_nf(predict_X_raw, has_y=False)

# Compute how many hours per location we need to forecast
h_final = int(predict_nf['unique_id'].value_counts().min())
print(f'Prediction horizon (hours): {h_final}  (~{h_final/24:.1f} days)')

No Y_col: Sum of кВт
Prediction horizon (hours): 5089  (~212.0 days)


In [15]:
all_data_nf = (
    pd.concat([train_nf, val_nf, test_nf])
    .sort_values(['unique_id', 'ds'])
    .reset_index(drop=True)
)

# Trim for training (dataset init is the bottleneck); keep full all_data_nf for prediction context
all_data_trimmed = trim_nf(all_data_nf)
print(f'all_data_nf: {all_data_nf.shape}  →  trimmed: {all_data_trimmed.shape}')

# Train with h=HORIZON (1 month); recursive steps will handle the full 6-month window
nf_final = NeuralForecast(models=[make_nhits(HORIZON)], freq='h')
nf_final.fit(df=all_data_trimmed, static_df=static_df, val_size=HORIZON)

C:\Users\Lev\AppData\Local\Temp\ipykernel_13392\1081633619.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby('unique_id', group_keys=False)
Seed set to 1


all_data_nf: (5472975, 25)  →  trimmed: (1757201, 25)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ SMAPE         │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ blocks       │ ModuleList    │ 16.9 M │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 16.9 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 16.9 M                                                                                               
Total estimated model params size (MB): 67                                                                         
Modules in train mode: 94                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

IndexError: index 410 is out of bounds for dimension 0 with size 410

In [ ]:
# --- Recursive 6-month forecast: roll forward one HORIZON at a time ---
predict_futr = predict_nf[['unique_id', 'ds'] + FUTR_EXOG]

context_nf   = all_data_nf.copy()
monthly_preds = []

max_rows = predict_futr.groupby('unique_id').size().max()
n_steps  = int(np.ceil(max_rows / HORIZON))
print(f'Total steps: {n_steps}  ({HORIZON}h each)')

for step in range(n_steps):
    lo, hi = step * HORIZON, (step + 1) * HORIZON

    step_futr = (
        predict_futr
        .groupby('unique_id', group_keys=False)
        .apply(lambda g: g.iloc[lo:hi])
        .reset_index(drop=True)
    )
    if step_futr.empty:
        break

    # Predict the next HORIZON hours using the current context
    step_pred = nf_final.predict(df=context_nf, futr_df=step_futr)
    monthly_preds.append(step_pred.copy())

    # Append predicted values back into context for the next step
    new_rows = step_futr.merge(
        step_pred.rename(columns={'NHITS': 'y'})[['unique_id', 'ds', 'y']],
        on=['unique_id', 'ds'],
    )
    context_nf = (
        pd.concat([context_nf, new_rows[context_nf.columns]])
        .sort_values(['unique_id', 'ds'])
        .reset_index(drop=True)
    )
    print(f'  Step {step + 1}/{n_steps} done — context rows: {len(context_nf)}')

# Combine all steps and restore original column names
final_preds = (
    pd.concat(monthly_preds)
    .sort_values(['unique_id', 'ds'])
    .reset_index(drop=True)
    .rename(columns={'unique_id': ID_COL, 'ds': DS_COL, 'NHITS': Y_COL})
)
print(final_preds.shape)
final_preds.head(10)

In [ ]:
import os
os.makedirs('predictions', exist_ok=True)
final_preds.to_parquet('predictions/nhits_6month.parquet', index=False)
print('Saved to predictions/nhits_6month.parquet')